<a href="https://colab.research.google.com/github/Sashi09/social-media-insights-plotly/blob/main/Interactive_Social_Media_Insights_with_Plotly.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plotly

Plotly is an open-source Python library that facilitates the creation of interactive, web-based visualizations. It integrates effortlessly with pandas, allowing users to produce plots directly from DataFrame and Series objects to enhance data analysis efficiency.

# Installation

In [ ]:
!pip install plotly
!pip install cufflinks

## Plotly Express in Python

*    Plotly Express is a high-level interface for creating interactive visualizations in Python.

*   It is built on top of Plotly and provides a simpler syntax for plotting.

*  Pandas integration allows direct plotting from DataFrame and Series objects.


*  Supports various chart types, including scatter plots, bar charts, line graphs, and more.



* Requires less code compared to Plotly Graph Objects, making it user-friendly.

# Importing Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from plotly.offline import iplot
import plotly as py
import plotly.tools as tls
import cufflinks as cf
%matplotlib inline

# Loading The Dataset

In [ ]:
import pandas as pd

In [ ]:
df = pd.read_csv('/content/Social Media Influencer Analytics.csv')

In [ ]:
df

In [ ]:
df['Device_Used'].info()

In [ ]:
df['Video_Length'].unique()

In [ ]:
df['Comments_Likes'].unique()

In [ ]:
df['Profile_Verified']

In [ ]:
df['Platform'].value_counts()
df['Post_Type'].value_counts()
df['Brand_Collaboration'].value_counts()
df['Sentiment'].value_counts()
df['Influencer_Tier'].value_counts()
df['Profile_Verified'].value_counts()

In [ ]:
df_selected = df[['User_ID', 'Followers']].copy()
if df_selected['Followers'].isnull().any():
    df_selected['Followers'] = df_selected['Followers'].fillna(df_selected['Followers'].median())
    print("Missing values in 'Followers' column filled with the median.")
else:
    print("No missing values found in 'Followers' column.")

if df_selected['User_ID'].nunique() > 20:
    data_subset = df_selected.nlargest(20, 'Followers')
    print("Limited the number of users to the top 20 with the highest follower counts.")
else:
    data_subset = df_selected
    print("No subsetting performed as the number of unique users is not greater than 20.")

display(data_subset.head())

# Preprocess the Data

In [ ]:
df.columns

In [ ]:
numeric_cols = ['Likes', 'Shares', 'Comments', 'Followers', 'Post_Length', 'Post_Reach',
                'Comments_Likes', 'Engaged_Users', 'Followers_Growth', 'Time_Spent_on_Post', 'Video_Views']
numeric_cols

In [ ]:
categorical_cols = ['Platform', 'Post_Type', 'Brand_Collaboration', 'Post_Time', 'Sentiment',
                    'Device_Used', 'Comments_Sentiment', 'Region', 'Influencer_Tier',
                    'Profile_Verified', 'Content_Type']
categorical_cols

In [ ]:
for col in numeric_cols:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

In [ ]:
df[categorical_cols].isnull().sum()

In [ ]:
print("Shape of the DataFrame:", df.shape)
print("\nData Types:")
print(df.info())

In [ ]:
df[categorical_cols].isnull().sum()

In [ ]:
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype in ['int64', 'float64']:

            median_val = df[col].median()
            df[col].fillna(median_val, inplace=True)
        elif df[col].dtype == 'object':

            mode_val = df[col].mode()[0]
            df[col].fillna(mode_val, inplace=True)
        else:
            pass
numerical_cols = ['     Likes', 'Shares', 'Comments', 'Followers', 'Post_Length', 'Post_Reach', 'Comments_Likes', 'Engaged_Users', 'Followers_Growth', 'Time_Spent_on_Post', 'Video_Views']
numerical_cols = [col for col in numerical_cols if col in df.columns]
for col in numerical_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower=lower_bound, upper=upper_bound)
df['Platform'] = df['Platform'].str.lower()
df['Platform'] = df['Platform'].replace({'instagram': 'instagram', 'insta': 'instagram'})
display(df.head())

In [ ]:
df['Avg_Engagement_per_Post'] = (df['     Likes'] + df['Shares'] + df['Comments']) / 3
def categorize_influencer(followers):
    if followers < 10000:
        return 'Micro'
    elif followers < 100000:
        return 'Small'
    elif followers < 500000:
        return 'Medium'
    elif followers < 1000000:
        return 'Macro'
    else:
        return 'Mega'

df['Influencer_Category'] = df['Followers'].apply(categorize_influencer)
def categorize_post_length(length):
    if length < 50:
        return 'Short'
    elif length < 200:
        return 'Medium'
    else:
        return 'Long'

df['Post_Length_Category'] = df['Post_Length'].apply(categorize_post_length)

df['Comments_to_Likes_Ratio'] = df['Comments'] / df['     Likes']


display(df.head())

# Feature Engineering

In [ ]:
for col in numeric_cols:
    if col in df.columns:
        before_min = df[col].min()
        before_max = df[col].max()

        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df[col] = np.clip(df[col], lower, upper)

        after_min = df[col].min()
        after_max = df[col].max()

        print(f"{col}:\n  Before min: {before_min}, max: {before_max}")
        print(f"  After  min: {after_min}, max: {after_max}\n")

In [ ]:
if 'Followers' in df.columns:
    def categorize_influencer(f):
        if f < 10_000: return 'Micro'
        elif f < 100_000: return 'Small'
        elif f < 500_000: return 'Medium'
        elif f < 1_000_000: return 'Macro'
        else: return 'Mega'
    df['Influencer_Category'] = df['Followers'].apply(categorize_influencer)
print(df[['Followers', 'Influencer_Category']].head())
print(df['Influencer_Category'].value_counts())

In [ ]:
if 'Platform' in df.columns:
    df['Platform'] = df['Platform'].str.lower().replace({'insta': 'instagram', 'fb': 'facebook'})
    print(df['Platform'].head())
    print(df['Platform'].unique())
    print(df['Platform'].value_counts())

## Data visualization

### Subtask:
Visualize the data using Plotly to explore relationships between variables.



Create the visualizations specified in the instructions using Plotly.



# Stacked Area Chart

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

dates = pd.date_range(start="2025-04-01", end="2025-04-30", freq="D")
n_days = len(dates)
sample_data = {
    "Post_Time": np.repeat(dates, 10),
    "Likes": np.random.randint(50, 500, n_days * 10),
    "Comments": np.random.randint(10, 100, n_days * 10),
    "Shares": np.random.randint(5, 50, n_days * 10),
}

df = pd.DataFrame(sample_data)

df["Post_Date"] = pd.to_datetime(df["Post_Time"]).dt.date
daily_data = df.groupby("Post_Date").agg({
    "Likes": "sum",
    "Comments": "sum",
    "Shares": "sum"
}).reset_index()

# Create the figure
fig = go.Figure()

# Add traces for Likes, Comments, and Shares
fig.add_trace(go.Scatter(
    x=daily_data["Post_Date"],
    y=daily_data["Likes"],
    mode="lines",
    name="Likes",
    stackgroup="one",
    line=dict(width=0.5),
    fillcolor="rgba(255, 99, 71, 0.5)"
))

fig.add_trace(go.Scatter(
    x=daily_data["Post_Date"],
    y=daily_data["Comments"],
    mode="lines",
    name="Comments",
    stackgroup="one",
    line=dict(width=0.5),
    fillcolor="rgba(65, 105, 225, 0.5)"
))

fig.add_trace(go.Scatter(
    x=daily_data["Post_Date"],
    y=daily_data["Shares"],
    mode="lines",
    name="Shares",
    stackgroup="one",
    line=dict(width=0.5),
    fillcolor="rgba(50, 205, 50, 0.5)"
))

# Update layout
fig.update_layout(
    title="Daily Social Media Engagement (April 2025)",
    xaxis_title="Date",
    yaxis_title="Engagement Count",
    xaxis=dict(tickformat="%b %d", tickangle=45),
    yaxis=dict(gridcolor="lightgray"),
    showlegend=True,
    template="plotly_white",
    hovermode="x unified"  # Unified tooltip
)

# Show the plot
fig.show()

# Sunbrust Chart

In [ ]:
import plotly.express as px # import the plotly.express module and alias it as 'px'

df = px.data.wind().query('direction == "N"')

In [ ]:
df

In [ ]:
sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], size=100),
    "Post_Type": np.random.choice(["Image", "Video", "Text"], size=100),
    "Likes": np.random.randint(50, 500, size=100),
}
df = pd.DataFrame(sample_data)

df_agg = df.groupby(["Platform", "Post_Type"])["Likes"].sum().reset_index()
fig = px.sunburst(
    df_agg,
    path=["Platform", "Post_Type"],
    values="Likes",
    color="Likes",
    color_continuous_scale="Viridis",
    title="Engagement (Likes) by Platform and Post Type"
)
fig.update_layout(
    template="plotly_white",
    margin=dict(t=50, l=25, r=25, b=25)
)
fig.show()

# Tree Maps

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], size=100),
    "Post_Type": np.random.choice(["Image", "Video", "Text"], size=100),
    "Likes": np.random.randint(50, 500, size=100),
}
df = pd.DataFrame(sample_data)

df_agg = df.groupby(["Platform", "Post_Type"])["Likes"].sum().reset_index()

fig = px.treemap(
    df_agg,
    path=[px.Constant("All Platforms"), "Platform", "Post_Type"],
    values="Likes",
    color="Likes",
    color_continuous_scale="Viridis",  # Color scale for Likes
    title="Engagement (Likes) by Platform and Post Type"
)
fig.update_layout(
    template="plotly_white",
    margin=dict(t=50, l=25, r=25, b=25)
)

# Show the plot
fig.show()

# 1D Distributions

In [ ]:
import plotly.express as px
df = px.data.carshare()

In [ ]:
df

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np
sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 100),
    "Likes": np.random.randint(50, 500, 100),
    "Post_Reach": np.random.randint(1000, 10000, 100),
    "Post_Type": np.random.choice(["Image", "Video", "Text"], 100),
    "Comments": np.random.randint(10, 100, 100),
    "Shares": np.random.randint(5, 50, 100),
}
df = pd.DataFrame(sample_data)
fig = px.histogram(
    df,
    x="Likes",
    y="Post_Reach",
    color="Platform",
    hover_data=df.columns,
    marginal="rug",
    title="Distribution of Likes with Post Reach by Platform"
)
fig.update_layout(
    template="plotly_white",
    xaxis_title="Likes",
    yaxis_title="Sum of Post Reach"
)
fig.show()

# Viloin Plot

In [ ]:
import plotly.express as px
df = px.data.wind()

In [ ]:
df

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 100),
    "Likes": np.random.randint(50, 500, 100),
}

df = pd.DataFrame(sample_data)

fig = px.violin(
    df,
    x="Likes",
    y="Platform",
    color="Platform",
    box=True,
    title="Distribution of Likes by Platform"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Likes",
    yaxis_title="Platform"
)

fig.show()

# Bubble Chart

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 100),
    "Likes": np.random.randint(50, 500, 100),
    "Post_Reach": np.random.randint(1000, 10000, 100),
    "Comments": np.random.randint(10, 100, 100),
    "Region": np.random.choice(["North America", "Europe", "Asia", "Other"], 100),
}

df = pd.DataFrame(sample_data)


fig = px.scatter(
    df,
    x="Likes",
    y="Post_Reach",
    size="Comments",
    color="Platform",
    hover_name="Region",
    title="Social Media Engagement Scatter Plot"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Likes",
    yaxis_title="Post Reach"
)
import plotly.express as px
import pandas as pd
import numpy as np


sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 100),
    "Likes": np.random.randint(50, 500, 100),
    "Post_Reach": np.random.randint(1000, 10000, 100),
    "Comments": np.random.randint(10, 100, 100),
    "Region": np.random.choice(["North America", "Europe", "Asia", "Other"], 100),
}


df = pd.DataFrame(sample_data)

fig = px.scatter(
    df,
    x="Likes",
    y="Post_Reach",
    size="Comments",
    color="Platform",
    hover_name="Region",
    title="Social Media Engagement Scatter Plot"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Likes",
    yaxis_title="Post Reach"
)

fig.show()

# Animated Bubble Chart

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

dates = pd.date_range(start="2025-04-01", end="2025-04-30", freq="D")
sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 300),
    "Likes": np.random.randint(50, 500, 300),
    "Post_Reach": np.random.randint(1000, 10000, 300),
    "Comments": np.random.randint(10, 100, 300),
    "Region": np.random.choice(["North America", "Europe", "Asia", "Other"], 300),
    "Post_Time": np.random.choice(dates, 300),
}

df = pd.DataFrame(sample_data)

df["Post_Date"] = pd.to_datetime(df["Post_Time"]).dt.date

df_agg = df.groupby(["Platform", "Post_Date"]).agg({
    "Likes": "sum",
    "Post_Reach": "sum",
    "Comments": "sum",
    "Region": "first"
}).reset_index()

fig = px.scatter(
    df_agg,
    x="Likes",
    y="Post_Reach",
    size="Comments",
    color="Platform",
    hover_name="Region",
    animation_frame="Post_Date",
    title="Animated Social Media Engagement Over Time",
    range_x=[0, df_agg["Likes"].max() * 1.1],
    range_y=[0, df_agg["Post_Reach"].max() * 1.1],
    size_max=60
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Likes",
    yaxis_title="Post Reach",
    showlegend=True
)
fig.show()

# Density Heat Map

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 100),
    "Likes": np.random.randint(50, 500, 100),
}

df = pd.DataFrame(sample_data)


fig = px.density_heatmap(
    df,
    x="Likes",
    y="Platform",
    marginal_x="rug",
    marginal_y="histogram",
    title="Density Heatmap of Likes by Platform"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Likes",
    yaxis_title="Platform"
)

fig.show()


# 3D Coordinates

In [ ]:
df = px.data.wind()

In [ ]:
df

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 100),
    "Likes": np.random.randint(50, 500, 100),
    "Post_Reach": np.random.randint(1000, 10000, 100),
    "Comments": np.random.randint(10, 100, 100),
    "Engaged_Users": np.random.randint(50, 1000, 100),
    "Region": np.random.choice(["North America", "Europe", "Asia", "Other"], 100),
}

df = pd.DataFrame(sample_data)

fig = px.scatter_3d(
    df,
    x="Likes",
    y="Post_Reach",
    z="Comments",
    color="Platform",
    size="Engaged_Users",
    symbol="Platform",
    hover_name="Region",
    title="3D Scatter Plot of Social Media Engagement"
)


fig.update_layout(
    template="plotly_white",
    scene=dict(
        xaxis_title="Likes",
        yaxis_title="Post Reach",
        zaxis_title="Comments"
    )
)
fig.show()

In [ ]:
import plotly.express as px
fig = px.histogram(
    df,
    x='Likes',
    nbins=30,
    title="Distribution of Likes",
    color_discrete_sequence=['skyblue'],
    histnorm='probability density'
)
fig.show()

In [ ]:
import plotly.express as px
import pandas as pd

df = pd.read_csv('/content/Social Media Influencer Analytics.csv')
df['Avg_Engagement_per_Post'] = (df['     Likes'] + df['Shares'] + df['Comments']) / 3
def categorize_influencer(followers):
    if followers < 10000:
        return 'Micro'
    elif followers < 100000:
        return 'Small'
    elif followers < 500000:
        return 'Medium'
    elif followers < 1000000:
        return 'Macro'
    else:
        return 'Mega'

df['Influencer_Category'] = df['Followers'].apply(categorize_influencer)

fig = px.histogram(df, x='Influencer_Tier', color='Influencer_Tier', color_discrete_sequence=px.colors.sequential.Blues)
fig.update_layout(
    title='Count by Influencer Tier',
    xaxis_title='Influencer Tier',
    yaxis_title='Count'
)
fig.show()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df['Hashtag_Count'] = df['Hashtags'].apply(lambda x: len(str(x).split(',')))
fig = px.histogram(df, x="Followers", nbins=30,
                   title="Followers Distribution",
                   labels={"Followers": "Number of Followers"},
                   color_discrete_sequence=['skyblue'])
fig.update_layout(bargap=0.1, xaxis_title="Number of Followers", yaxis_title="Number of Influencers")
fig.show()

fig = px.histogram(df, x="Hashtag_Count",
                   title="Distribution of Hashtag Count per Post",
                   labels={"Hashtag_Count": "Number of Hashtags"},
                   color_discrete_sequence=['coral'])
fig.update_layout(bargap=0.1, xaxis_title="Number of Hashtags", yaxis_title="Number of Posts")
fig.show()

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from scipy.stats import gaussian_kde
import numpy as np

hist_color = '#8000ff'


for col in df.select_dtypes(include='number').columns:

    fig = px.histogram(
        df,
        x=col,
        title=f'Distribution of {col}',
        color_discrete_sequence=[hist_color],
        histnorm='probability density'
    )

    kde_data = df[col].replace([np.inf, -np.inf], np.nan).dropna()

    if len(kde_data) > 1 and kde_data.nunique() > 1:
        try:
            kde = gaussian_kde(kde_data, bw_method='scott')
            x_range = np.linspace(kde_data.min(), kde_data.max(), 100)
            kde_y = kde(x_range)

            fig.add_trace(go.Scatter(x=x_range, y=kde_y, mode='lines', name='KDE', line=dict(color='red')))
        except Exception as e:
            print(f"Warning: Skipping KDE for '{col}' due to error: {e}")
    else:
        print(f"Skipping KDE for '{col}' due to insufficient data or zero variance.")

    fig.update_layout(
        xaxis_title=col,
        yaxis_title='Probability Density',
        bargap=0.2
    )

    fig.show()

In [ ]:
import plotly.express as px

fig = px.histogram(df, x='Profile_Verified', color='Profile_Verified',
                   color_discrete_sequence=px.colors.qualitative.Set2,
                   title="Verified vs Non-Verified Profiles")
fig.update_layout(
    xaxis_title="Profile Verified",
    yaxis_title="Count",
    showlegend=False
)
fig.show()

In [ ]:
import pandas as pd

df.rename(columns=lambda x: x.strip().replace(" ", "_"), inplace=True)
x_column = 'Outlet' if 'Outlet' in df.columns else 'Platform'

fig = px.scatter(
    df,
    x=x_column,
    y='Post_Reach',
    color='Influencer_Tier',
    title='Outlet vs Post Reach',
    size_max=100,
    color_discrete_sequence=px.colors.qualitative.Set1,
    labels={x_column: 'Outlet', 'Post_Reach': 'Post Reach'}
)

fig.update_layout(
    width=800,
    height=480,
    xaxis_title='Outlet',
    yaxis_title='Post Reach',
    xaxis_tickangle=45,
    template='simple_white',
    legend_title='Influencer Tier'
)

fig.show()

# Box Plot

In [ ]:
import plotly.express as px

fig = px.box(df, x='Followers', title="Boxplot of Followers", width=1000, height=600) # Changed 'data' to 'df'
fig.update_traces(fillcolor='lightgreen', line=dict(color='lightgreen'), marker=dict(color='lightgreen'))
fig.update_layout(xaxis_title="Followers")
fig.show()

In [ ]:
import pandas as pd
import plotly.express as px

df = pd.read_csv("Social Media Influencer Analytics.csv")
df.columns = df.columns.str.strip()

cols_to_plot = ['Likes', 'Comments', 'Shares']

fig = px.box(df, y=cols_to_plot, title='Boxplot of Likes, Comments, and Shares', width=1000, height=600)

fig.update_layout(yaxis_title='Count')

fig.show()

In [ ]:
import pandas as pd
import plotly.express as px

top10_region = df['Region'].value_counts().head(10).sort_values(ascending=True)

fig = px.bar(
    top10_region,
    x=top10_region.values,
    y=top10_region.index,
    title='Top 10 Regions by Number of Outlets',
    labels={'x': 'Number of Outlets', 'y': 'Region'},
    color=top10_region.values,
    color_continuous_scale='Blues'
)
fig.update_layout(barmode='stack', xaxis_title='Number of Outlets', yaxis_title='Region')
fig.show()

In [ ]:
import plotly.express as px
df_subset = df.sort_values(by='Followers', ascending=False).head(20)

fig = px.bar(df_subset, x='User_ID', y='Followers',
             title='Number of Followers per User ID (Top 20)',
             labels={'User_ID': 'User ID', 'Followers': 'Followers'},
             color_discrete_sequence=['skyblue'])
fig.update_layout(
    xaxis_tickangle=90,
    width=800,
    height=400,
    margin=dict(l=0, r=0, t=50, b=0),
    showlegend=False
)
fig.show()

# Kernel Density Estimate (KDE) Plot

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd
import numpy as np
from scipy import stats

df.columns = df.columns.str.strip()
color_code = "#1f77b4"

kde = stats.gaussian_kde(df['Engaged_Users'])
x = np.linspace(0, 2000, 100)
y = kde(x)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=x,
    y=y,
    fill='tozeroy',
    mode='lines',
    line=dict(color=color_code),
    name='KDE'
))

fig.update_layout(
    title="Engaged Users Distribution",
    xaxis_title="Engaged Users",
    yaxis_title="Density",
    xaxis=dict(
        range=[0, 2000],
        tickmode='array',
        tickvals=list(range(0, 2001, 200))
    ),
    showlegend=False
)
fig.write_html('engaged_users_kde.html')
fig.show()

# Heat Map

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

df.columns = df.columns.str.strip()
numerical_data = df.select_dtypes(include='number')
corr_matrix = numerical_data.corr()

fig = go.Figure(go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmin=-1, zmax=1,
    text=corr_matrix.values,
    texttemplate="%{text:.2f}",
    textfont={"size": 10},
    colorbar=dict(title="Correlation")
))

fig.update_layout(
    title="Correlation Heatmap of Influencer Metrics",
    width=800,
    height=600,
    xaxis=dict(tickangle=45, side="top"),
    yaxis=dict(autorange="reversed"),
    margin=dict(l=100, r=100, t=150, b=100)
)
fig.show()

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
content_time_reach = df.groupby(['Content_Type', 'Post_Time'])['Post_Reach'].sum().reset_index()
heatmap_data = content_time_reach.pivot(index='Content_Type', columns='Post_Time', values='Post_Reach')
fig = go.Figure(data=go.Heatmap(
    z=heatmap_data.values,
    x=heatmap_data.columns,
    y=heatmap_data.index,
    colorscale='RdBu',
    text=heatmap_data.values,
    texttemplate="%{text:.0f}",
    textfont={"size": 10}
))
fig.update_layout(
    title='Post Reach by Content Type and Post Time',
    xaxis_title='Post Time',
    yaxis_title='Content Type',
    xaxis_tickangle=45,
    width=800,
    height=600
)
fig.show()

In [ ]:
fig = px.pie(df, values='Likes', names='Platform') # Remove leading spaces from '     Likes' to 'Likes'
fig.show()

In [ ]:
import plotly.express as px

content_counts = df['Content_Type'].value_counts().reset_index()
content_counts.columns = ['Content_Type', 'Count']

fig = px.pie(content_counts,
             values='Count',
             names='Content_Type',
             title='Distribution of Content Types',
             color_discrete_sequence=px.colors.qualitative.Set3)

fig.update_traces(textinfo='percent+label', pull=[0.05]*len(content_counts), rotation=90)
fig.update_layout(showlegend=True, margin=dict(t=50, b=50, l=50, r=50))
fig.show()

# Bar Graph

In [ ]:
import plotly.express as px

sentiment_counts = df['Sentiment'].value_counts().reset_index()
sentiment_counts.columns = ['Sentiment', 'Count']

fig = px.bar(sentiment_counts,
             x='Sentiment',
             y='Count',
             title='Count of Sentiments',
             color='Sentiment',
             color_discrete_sequence=px.colors.sequential.Rainbow)

fig.update_layout(showlegend=False, xaxis_title='Sentiment', yaxis_title='Count')
fig.show()

# Bi-Directional Graph

In [ ]:
import plotly.graph_objects as go
import plotly.io as pio
region_counts = df['Region'].value_counts().head(10).sort_values()
left = region_counts[::2] * -1
right = region_counts[1::2]

fig = go.Figure()
fig.add_trace(go.Bar(
    y=left.index,
    x=left.values,
    orientation='h',
    name='Category A',
    marker_color='salmon'
))

fig.add_trace(go.Bar(
    y=right.index,
    x=right.values,
    orientation='h',
    name='Category B',
    marker_color='skyblue'
))

fig.update_layout(
    title='Bi-directional Bar Chart for Region Distribution',
    xaxis_title='Number of Outlets',
    showlegend=True,
    barmode='relative',
    shapes=[dict(
        type='line',
        x0=0,
        x1=0,
        y0=0,
        y1=1,
        yref='paper',
        line=dict(color='black', width=1)
    )],
    width=800,
    height=600
)

pio.show(fig)

# Scatter Plot

In [ ]:
import plotly.express as px
import pandas as pd
import numpy as np

sample_data = {
    "Platform": np.random.choice(["Instagram", "Twitter", "Facebook", "TikTok"], 100),
    "Likes": np.random.randint(50, 500, 100),
    "Post_Reach": np.random.randint(1000, 10000, 100),
}


df = pd.DataFrame(sample_data)


fig = px.scatter_matrix(
    df,
    dimensions=["Likes", "Post_Reach"],
    color="Platform",
    title="Scatter Matrix of Social Media Engagement Metrics"
)

fig.update_layout(template="plotly_white")


fig.show()

In [ ]:
import plotly.express as px
import plotly.io as pio

fig = px.box(
    df,
    x='Platform',
    y='Post_Reach',
    color='Platform',
    color_discrete_sequence=px.colors.qualitative.Set2,
    title='Platform vs Post Reach'
)


fig.update_layout(
    xaxis_title='Platform',
    yaxis_title='Post Reach',
    width=800,
    height=500,
    showlegend=False,
    xaxis_tickangle=45
)
pio.show(fig)

# Grouped Bar chart

In [ ]:
import plotly.express as px

fig = px.histogram(df, x='Sentiment', color='Platform', barmode='group',
                   color_discrete_sequence=px.colors.qualitative.Pastel,
                   title='Sentiment Distribution by Platform',
                   labels={'Sentiment': 'Sentiment', 'count': 'Count'})
fig.update_layout(width=800, height=500, showlegend=True)
fig.show()

# Violin Plot

In [ ]:
import plotly.express as px
import plotly.io as pio

fig = px.violin(
    df,
    x='Content_Type',
    y='Post_Reach',
    color='Influencer_Tier',
    violinmode='overlay',
    title='Distribution of Post Reach by Content Type and Influencer Tier',
    color_discrete_sequence=px.colors.qualitative.Set2
)

fig.update_layout(
    xaxis_title='Content Type',
    yaxis_title='Post Reach',
    xaxis_tickangle=45,
    autosize=False,
    width=800,
    height=480,
    margin=dict(l=40, r=40, t=60, b=100)
)

In [ ]:
import plotly.express as px
df.columns = df.columns.str.strip()
fig = px.scatter_matrix(
    df,
    dimensions=df.select_dtypes(include=['float64', 'int64']).columns,
    title='Pairplot of Numeric Features'
)
fig.update_layout(
    width=1000,
    height=1000,
    title_y=0.98
)
fig.show()

# Final Insights & Conclusion


* Instagram, TikTok, and Facebook emerged as the most engaging platforms based on total likes across the dataset.

*   The distribution of followers varies significantly by platform, with some exhibiting highly skewed audience reach.
* Interactive box plots helped uncover patterns and outliers in follower counts, enhancing visibility into influencer reach.

* Data cleaning was essential—handling null values, standardizing text columns, and normalizing platform names ensured reliable analysis.

* Likes, shares, and comments remain core engagement metrics, with certain platforms consistently outperforming others in these areas.


*   Grouping and ranking platforms by total likes allowed easy identification of high-performing social media outlets.
*   This analysis offers a solid foundation for brands, marketers, and content creators to understand platform performance and audience behavior.

This EDA provides an interactive, visual-driven snapshot of social media trends—helping decode where engagement thrives and which platforms hold the most influence.